In [10]:
import os
from dotenv import load_dotenv

# 1. Le preguntamos a Python en qué carpeta está parado
carpeta_actual = os.getcwd()
print(f"Python está buscando en esta carpeta: {carpeta_actual}")

# 2. Forzamos a que lea el archivo y sobreescriba la memoria
load_dotenv(override=True)

# 3. Intentamos leer de nuevo
api_key = os.getenv("LLM_API_KEY")

if api_key:
    print(" API Key cargada correctamente.")
else:
    print(" API NO ENCONTRADA. Fijarse si la carpeta de arriba es donde guardaste el .env, o si lo guardaste bien al archivo")

Python está buscando en esta carpeta: c:\Users\Santi\Desktop\Proyecto_integrador_bases_de_datos\IAJupyter
 API Key cargada correctamente.


In [11]:
import os
import mysql.connector
from dotenv import load_dotenv

# Cargamos las variables (por si acaso)
load_dotenv()

try:
    # 1. Intentamos establecer la conexión usando los datos del .env
    conexion = mysql.connector.connect(
        host=os.getenv("DB_HOST"),
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
        database=os.getenv("DB_NAME")
    )
    
    if conexion.is_connected():
        print("¡Conexión exitosa a la base de datos!")
        
        # 2. Creamos un 'cursor' (es el mensajero que lleva el SQL y trae los datos)
        cursor = conexion.cursor()
        
        # 3. Probamos ejecutar una consulta súper básica
        cursor.execute("SHOW TABLES;")
        
        # 4. Obtenemos los resultados y los imprimimos
        tablas = cursor.fetchall()
        print("\nTablas y vistas encontradas en tu base de datos:")
        for tabla in tablas:
            print(f"{tabla[0]}")
            
        # Cerramos el cursor
        cursor.close()

except mysql.connector.Error as error:
    # Si hay un error (clave mal, motor apagado), nos avisa acá
    print(f"❌ Error al conectar a MySQL: {error}")

finally:
    # 5. Siempre es buena práctica cerrar la conexión al terminar
    if 'conexion' in locals() and conexion.is_connected():
        conexion.close()
        print("\n Conexión cerrada correctamente.")

¡Conexión exitosa a la base de datos!

Tablas y vistas encontradas en tu base de datos:
auditoria_prestamos
autor
ejemplar
genero
libro
libro_autor
libro_genero
prestamo
sancion
socio
vw_libros_disponibles
vw_prestamos_activos
vw_socios_sancionados

 Conexión cerrada correctamente.


In [12]:
import os
from google import genai
from dotenv import load_dotenv

# Cargamos la clave del .env
load_dotenv()
api_key = os.getenv("LLM_API_KEY")

# 1. Inicializamos el cliente con la nueva librería
client = genai.Client(api_key=api_key)

try:
    print("Enviando mensaje a la IA con la nueva librería")
    
    # 2. Le mandamos el mensaje usando el modelo más reciente (gemini-2.5-flash)
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents='Hola! Vamos a trabajar armando consultas SQL para una biblioteca. Respondeme en una sola oración corta si estás listo.'
    )
    
    # 3. Imprimimos lo que nos contestó
    print("\n ¡Conexión exitosa con la IA!")
    print(f"Gemini dice: {response.text}")

except Exception as e:
    print(f"\n Error al conectar con la IA: {e}")

Enviando mensaje a la IA con la nueva librería

 ¡Conexión exitosa con la IA!
Gemini dice: ¡Listo!


In [15]:
import os
import mysql.connector
from google import genai
from dotenv import load_dotenv
import pandas as pd
from google.genai import types
import warnings
from IPython.display import display

# Desactivamos el aviso rojo de Pandas
warnings.filterwarnings('ignore', category=UserWarning)

# Cargamos variables de entorno
load_dotenv()
api_key = os.getenv("LLM_API_KEY")
client = genai.Client(api_key=api_key)

def ejecutar_consulta_ia(pregunta_usuario):
    print(f"🤔 Procesando pregunta: '{pregunta_usuario}'...\n")
    
    # 1. Esquema limpio y preciso de tu base de datos
    esquema_base_datos = """
    Tablas MySQL disponibles:
    - GENERO (id_genero, nombre, descripcion)
    - AUTOR (id_autor, nombre, apellido, nacionalidad)
    - LIBRO (isbn, titulo, anio_publicacion, stock_total, stock_disponible)
    - SOCIO (id_socio, dni, nombre, apellido, email, fecha_alta, estado)
    - LIBRO_AUTOR (isbn, id_autor)
    - LIBRO_GENERO (isbn, id_genero)
    - EJEMPLAR (id_ejemplar, isbn, nro_ejemplar, estado_fisico)
    - SANCION (id_sancion, id_socio, tipo, fecha_inicio, fecha_fin, motivo)
    - PRESTAMO (id_prestamo, id_socio, id_ejemplar, fecha_prestamo, fecha_vencimiento, fecha_devolucion, estado)

    Vistas disponibles (usar preferentemente si aplica a la pregunta):
    - vw_prestamos_activos (id_prestamo, nombre_socio, email_socio, titulo_libro, fecha_vencimiento, dias_de_mora)
    - vw_libros_disponibles (isbn, titulo, autor, stock_disponible)
    - vw_socios_sancionados (id_socio, nombre_socio, estado_actual, total_sanciones_historicas)
    """
    
    # 2. Instrucciones directas e inquebrantables
    system_instruction = """
    Sos un traductor de lenguaje natural a SQL para MySQL. Tu única salida debe ser código SQL ejecutable.
    Cero explicaciones, cero saludos, cero formato Markdown (no uses ```sql ni ```).
    
    Reglas relacionales estrictas:
    1. LIBRO y AUTOR se unen OBLIGATORIAMENTE pasando por la tabla intermedia LIBRO_AUTOR.
    2. LIBRO y GENERO se unen OBLIGATORIAMENTE pasando por la tabla intermedia LIBRO_GENERO.
    3. En AUTOR y SOCIO, para buscar por nombre y apellido a la vez, usa: CONCAT(nombre, ' ', apellido).
    """
    
    # 3. Llamada a la IA y ejecución
    try:
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=f"{esquema_base_datos}\nPregunta del usuario: {pregunta_usuario}",
            config=types.GenerateContentConfig(
                system_instruction=system_instruction,
                temperature=0.1
            )
        )
        
        sql_query = response.text.strip()
        
        # Limpieza de seguridad por si manda bloques markdown
        if sql_query.startswith("```sql"): 
            sql_query = sql_query[6:]
        if sql_query.startswith("```"): 
            sql_query = sql_query[3:]
        if sql_query.endswith("```"): 
            sql_query = sql_query[:-3]
            
        sql_query = sql_query.strip()
            
        print(f"🤖 SQL generado por la IA:\n{sql_query}\n")
        
        # Conexión a la BD
        conexion = mysql.connector.connect(
            host=os.getenv("DB_HOST"),
            user=os.getenv("DB_USER"),
            password=os.getenv("DB_PASSWORD"),
            database=os.getenv("DB_NAME")
        )
        
        if conexion.is_connected():
            df = pd.read_sql(sql_query, con=conexion)
            return df
            
    except Exception as e:
        print(f"❌ Ocurrió un error: {e}")
        return None
        
    finally:
        if 'conexion' in locals() and conexion.is_connected():
            conexion.close()

In [14]:
# Hacé la pregunta que quieras en español
pregunta = "que libros hay hechos por julio cortazar?"

resultado = ejecutar_consulta_ia(pregunta)

if resultado is not None and not resultado.empty:
    print("📊 Resultados obtenidos:")
    display(resultado) # display() en Jupyter muestra un DataFrame como una tabla interactiva
else:
    print("📭 No se encontraron resultados o la consulta falló.")

🤔 Procesando pregunta: 'que libros hay hechos por julio cortazar?'...

🤖 SQL generado por la IA:
SELECT L.titulo
FROM LIBRO L
JOIN LIBRO_AUTOR LA ON L.isbn = LA.isbn
JOIN AUTOR A ON LA.id_autor = A.id_autor
WHERE CONCAT(A.nombre, ' ', A.apellido) = 'Julio Cortazar'

📊 Resultados obtenidos:


,titulo
0,Rayuela
1,Bestiario
